# 📧 Spam Email Detector
### Machine Learning Project — XGBoost Classifier
---
This notebook trains a spam detection model using XGBoost and RandomizedSearchCV on a dataset of 5172 emails.

## 1. 📦 Imports

In [ ]:
import numpy as np
import pandas as pd
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import classification_report, confusion_matrix

## 2. 📊 Load Dataset

In [ ]:
df = pd.read_csv('emails.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## 3. ✂️ Prepare Features and Labels

In [ ]:
X = df.drop(['Email No.', 'Prediction'], axis=1)
y = df['Prediction']

print(f"X shape: {X.shape}")
print(f"Spam count: {y.sum()} | Not Spam count: {(y == 0).sum()}")

## 4. ✂️ Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape[0]} emails")
print(f"Test size:  {X_test.shape[0]} emails")

## 5. 🔧 Hyperparameter Tuning with RandomizedSearchCV

In [ ]:
xgb = XGBClassifier(random_state=42)

gbm_param_grid = {
    'learning_rate': np.arange(0.05, 1.05, .05),
    'n_estimators': np.arange(50, 200, 50),
    'max_depth': [3, 4, 5, 6, 7, 8, 9],
    'subsample': np.arange(0.05, 1.05, .05),
    'colsample_bytree': np.arange(0.05, 1.05, .05),
    'reg_alpha': [0, 0.1, 0.5, 1],
    'reg_lambda': [1, 1.5, 2, 5]
}

model = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=gbm_param_grid,
    n_iter=200,
    scoring='f1',
    cv=4,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

## 6. 📈 Results

In [ ]:
print("Best params:", model.best_params_)
print("Best F1 score:", model.best_score_)

best_model = model.best_estimator_
y_pred = best_model.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## 7. 💾 Save Model and Data

In [ ]:
# Save X_test and y_test for Streamlit app
X_test.to_csv('X_test.csv', index=False)
print("X_test saved ✅")

y_test.to_csv('y_test.csv', index=False)
print("y_test saved ✅")

# Save model and feature names
joblib.dump(best_model, 'spam_model1.pkl')
joblib.dump(X.columns.tolist(), 'feature_names1.pkl')
print("Model saved ✅")